In [1]:
import json
import time
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
from joblib import dump

warnings.filterwarnings("ignore")

In [2]:


# -------- CONFIG --------
DATA_PATH = "/content/credit_risk.csv"   # update path if needed
TARGET_COL = "Default"
OUT_DIR = Path("/mnt/data/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_RANDOM_ITER = 6
CV_FOLDS = 3
N_JOBS = 1
PACKAGE_ZIP = Path("/mnt/data/submission_package.zip")


# -------- HELPERS --------
def detect_target_column(df):
    commons = ['target', 'TARGET', 'label', 'y', 'Y', 'default', 'default_flag', 'is_default', 'Default']
    for name in commons:
        for c in df.columns:
            if c.lower() == name.lower():
                return c
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique(dropna=True) == 2:
            return c
    return df.columns[-1]


def safe_savefig(path):
    try:
        plt.savefig(path, bbox_inches="tight")
    except Exception:
        plt.savefig(path)
    plt.clf()


def build_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    onehot_kwargs = {}
    try:
        # sklearn >= 1.2 uses sparse_output
        OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        onehot_kwargs["sparse_output"] = False
    except TypeError:
        # older sklearn
        onehot_kwargs["sparse"] = False

    if cat_cols:
        categorical_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", **onehot_kwargs))
        ])
        transformers = [("num", numeric_transformer, num_cols),
                        ("cat", categorical_transformer, cat_cols)]
    else:
        transformers = [("num", numeric_transformer, num_cols)]

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    return preprocessor, num_cols, cat_cols


# -------- MAIN --------
def main():
    t0_total = time.time()

    if not Path(DATA_PATH).exists():
        raise FileNotFoundError(f"Data not found: {DATA_PATH}")

    df = pd.read_csv(DATA_PATH)
    print("Loaded:", df.shape)

    target = TARGET_COL if TARGET_COL in df.columns else detect_target_column(df)
    print("Using target:", target)

    df = df.copy()
    df["_orig_index"] = range(len(df))

    # map target to 0/1 if necessary
    y_ser = df[target]
    if (not pd.api.types.is_numeric_dtype(y_ser)) or (set(y_ser.dropna().unique()) - {0, 1}):
        uniques = list(pd.Series(y_ser.dropna().unique()))
        lowered = [str(u).strip().lower() for u in uniques]
        mapping = {}
        if set(lowered) <= {"yes", "no"} or set(lowered) <= {"y", "n"}:
            for u in uniques:
                mapping[u] = 1 if str(u).strip().lower() in ("yes", "y") else 0
        else:
            uniq_sorted = sorted([str(u) for u in uniques])
            mapping = {u: i for i, u in enumerate(uniq_sorted)}
        df["_original_target"] = df[target]
        df[target] = df[target].map(lambda v: mapping.get(v, mapping.get(str(v), v)))
        with open(OUT_DIR / "target_mapping.json", "w") as f:
            json.dump(mapping, f, indent=2)

    X = df.drop(columns=[target])
    y = df[target].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    print("Train/test:", X_train.shape, X_test.shape)

    preprocessor, num_cols, cat_cols = build_preprocessor(X)
    print(f"Numeric cols: {len(num_cols)}  Categorical cols: {len(cat_cols)}")

    # Choose model (XGBoost if available)
    use_xgb = False
    try:
        from xgboost import XGBClassifier
        use_xgb = True
        print("XGBoost available.")
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        print("XGBoost not available; using HistGradientBoostingClassifier.")

    if use_xgb:
        clf = XGBClassifier(
            objective="binary:logistic",
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS
        )
        pipeline = Pipeline([("pre", preprocessor), ("clf", clf)])
        param_dist = {
            "clf__n_estimators": [50, 100],
            "clf__max_depth": [3, 4],
            "clf__learning_rate": [0.03, 0.05, 0.1],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.8, 1.0]
        }
    else:
        clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        pipeline = Pipeline([("pre", preprocessor), ("clf", clf)])
        param_dist = {
            "clf__max_iter": [100, 150],
            "clf__max_leaf_nodes": [31, 63],
            "clf__learning_rate": [0.03, 0.05, 0.1],
            "clf__min_samples_leaf": [20, 50]
        }

    # Hyperparameter tuning
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    rs = RandomizedSearchCV(
        pipeline, param_dist, n_iter=N_RANDOM_ITER, scoring="roc_auc",
        cv=cv, n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=1, refit=True
    )
    print("Running RandomizedSearchCV ...")
    t0 = time.time()
    rs.fit(X_train, y_train)
    t_search = time.time() - t0
    print("Search done. Best CV roc_auc:", rs.best_score_)
    best_pipeline = rs.best_estimator_

    # -------- Predictions & metrics --------
    y_pred = best_pipeline.predict(X_test)
    try:
        y_proba = best_pipeline.predict_proba(X_test)[:, 1]
    except Exception:
        # some classifiers may not implement predict_proba
        y_proba = None

    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)) if y_proba is not None else None,
        "tuning_time_s": float(t_search)
    }
    with open(OUT_DIR / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("Metrics saved.")

    pred_df = X_test.reset_index(drop=True).copy()
    pred_df["y_true"] = y_test.reset_index(drop=True)
    pred_df["y_pred"] = y_pred
    pred_df["y_proba"] = y_proba if y_proba is not None else np.nan
    pred_df.to_csv(OUT_DIR / "predictions.csv", index=False)

    dump(best_pipeline, OUT_DIR / "model_pipeline.joblib")
    with open(OUT_DIR / "best_params.json", "w") as f:
        json.dump(rs.best_params_, f, indent=2)

    # ROC
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.figure(figsize=(6, 4))
        plt.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.4f}")
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend(loc="lower right")
        safe_savefig(PLOTS_DIR / "roc_curve.png")

    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
    plt.title("Confusion Matrix")
    safe_savefig(PLOTS_DIR / "confusion_matrix.png")

    # -------- SHAP explainability (global + local) --------
    shap_ok = False
    try:
        import shap
        shap_ok = True
        print("SHAP available. Running SHAP analysis (may be slow).")
    except Exception:
        print("SHAP not available; skipping SHAP plots and narratives. Install shap to enable full deliverables.")

    top_features = []
    shap_text_lines = []
    local_narratives = []

    if shap_ok:
        try:
            pre = best_pipeline.named_steps["pre"]
            model = best_pipeline.named_steps["clf"]

            # Transform train and test
            X_train_trans = pre.transform(X_train)
            X_test_trans = pre.transform(X_test)

            # Build feature names list
            feature_names = []
            if len(num_cols) > 0:
                feature_names.extend(num_cols)

            if len(cat_cols) > 0:
                try:
                    ohe = pre.named_transformers_["cat"].named_steps["onehot"]
                    try:
                        # sklearn >= 1.0
                        cat_names = list(ohe.get_feature_names_out(cat_cols))
                    except Exception:
                        # fallback to categories_
                        cats = getattr(ohe, "categories_", None)
                        if cats is not None:
                            cat_names = []
                            for c, vals in zip(cat_cols, cats):
                                for v in vals:
                                    cat_names.append(f"{c}__{v}")
                        else:
                            # fallback: create placeholder names
                            cat_names = [f"cat_{i}" for i in range(X_train_trans.shape[1] - len(feature_names))]
                    feature_names.extend(cat_names)
                except Exception:
                    # no categorical transformer / failed to extract names
                    pass

            # If anything mismatched, fallback to generic names
            if len(feature_names) != X_train_trans.shape[1]:
                feature_names = [f"f_{i}" for i in range(X_train_trans.shape[1])]

            # Create SHAP explainer
            # For tree models, TreeExplainer is often fastest; shap.Explainer will choose automatically.
            explainer = shap.Explainer(model, X_train_trans)
            shap_res = explainer(X_train_trans)
            shap_values = shap_res.values  # could be (n, features) or (n, outputs, features)

            # normalize shap_values shape to (n_samples, n_features)
            sv = shap_values
            if getattr(sv, "ndim", 1) == 3:
                # pick positive class index 1 when present
                sv_plot = sv[:, 1, :] if sv.shape[1] > 1 else sv[:, 0, :]
            else:
                sv_plot = sv

            # Global summary plots
            try:
                shap.summary_plot(sv_plot, X_train_trans, feature_names=feature_names, show=False)
                safe_savefig(PLOTS_DIR / "shap_summary_dot.png")
            except Exception as e:
                print("SHAP summary dot failed:", e)

            try:
                shap.summary_plot(sv_plot, X_train_trans, feature_names=feature_names, plot_type="bar", show=False)
                safe_savefig(PLOTS_DIR / "shap_summary_bar.png")
            except Exception as e:
                print("SHAP summary bar failed:", e)

            # Top features by mean absolute SHAP
            mean_abs = np.abs(sv_plot).mean(axis=0)
            top_idx = np.argsort(mean_abs)[::-1][:10]  # top 10
            top_features = [feature_names[i] for i in top_idx.tolist()]

            # For each top feature, create dependence plot (only top 5 to save time)
            for feat in top_features[:5]:
                try:
                    shap.dependence_plot(feat, sv_plot, X_train_trans, feature_names=feature_names, show=False)
                    safe_savefig(PLOTS_DIR / f"shap_dependence_{feat}.png")
                except Exception as e:
                    print("dependence plot failed for", feat, e)

            # Compute global interpretation text: feature direction
            # direction = mean SHAP (positive increases predicted risk)
            mean_signed = sv_plot.mean(axis=0)
            global_interpret = []
            for idx in top_idx[:10]:
                feat = feature_names[idx]
                mean_sh = mean_signed[idx]
                direction = "increases" if mean_sh > 0 else "decreases"
                global_interpret.append((feat, float(mean_abs[idx]), float(mean_sh), direction))

            # Compose textual summary
            shap_text_lines.append("### Global SHAP summary - top features and impact direction")
            shap_text_lines.append("")
            shap_text_lines.append("Top features (by mean |SHAP|):")
            for feat, mag, mean_sh, direction in global_interpret:
                shap_text_lines.append(f"- {feat}: mean |SHAP|={mag:.4f}, mean SHAP={mean_sh:.4f} → {direction} default risk")

            # Local explanations: pick one high-risk and one low-risk from test set
            # ensure we have y_proba; if not, use model decision_function or predictions and probability-like ordering
            if y_proba is None:
                # try decision_function
                try:
                    y_score = best_pipeline.decision_function(X_test)
                    order = np.argsort(y_score)
                except Exception:
                    # fallback to predicted labels
                    order = np.argsort(pred_df["y_pred"].values)
            else:
                order = np.argsort(pred_df["y_proba"].values)

            local_indices = []
            if len(order) >= 2:
                low_i = int(order[0])
                high_i = int(order[-1])
                local_indices = [low_i, high_i]
            elif len(order) == 1:
                local_indices = [int(order[0])]

            # transform test for indexing
            X_test_trans = X_test_trans if 'X_test_trans' in locals() else pre.transform(X_test)

            # build local narratives
            for k, idx in enumerate(local_indices):
                try:
                    inst_trans = X_test_trans[idx: idx + 1]
                    local_exp = explainer(inst_trans)
                    # sv for this instance (choose positive class slice if needed)
                    vals = local_exp.values
                    if getattr(vals, "ndim", 1) == 3:
                        vals_plot = vals[:, 1, :] if vals.shape[1] > 1 else vals[:, 0, :]
                        vals_plot = vals_plot[0]
                    else:
                        vals_plot = vals[0]

                    # identify top contributing features (positive -> pushing toward default, negative -> against)
                    contrib_order = np.argsort(np.abs(vals_plot))[::-1]
                    top_pos = [feature_names[i] for i in contrib_order if vals_plot[i] > 0][:5]
                    top_neg = [feature_names[i] for i in contrib_order if vals_plot[i] < 0][:5]

                    # Save a waterfall plot (preferred) for the instance
                    try:
                        shap.plots.waterfall(local_exp[0], show=False)
                        safe_savefig(PLOTS_DIR / f"shap_local_waterfall_{k}.png")
                    except Exception:
                        # fallback to force_plot (matplotlib)
                        try:
                            base_vals = getattr(local_exp, "base_values", None)
                            vals_local = getattr(local_exp, "values", None)
                            if base_vals is not None and vals_local is not None:
                                shap.force_plot(base_vals, vals_local[0], inst_trans, feature_names=feature_names, matplotlib=True, show=False)
                                safe_savefig(PLOTS_DIR / f"shap_local_force_{k}.png")
                        except Exception as exf:
                            print("local SHAP plot fallback failed:", exf)

                    # Build narrative for non-technical audience
                    prob_text = f"predicted probability (approx): {float(pred_df['y_proba'].iloc[idx]):.4f}" if 'y_proba' in locals() and not np.isnan(pred_df['y_proba'].iloc[idx]) else f"predicted label: {int(pred_df['y_pred'].iloc[idx])}"
                    narrative_lines = [
                        f"Case {k+1} (test index {idx}) — {prob_text}",
                        "Top features increasing default risk: " + (", ".join(top_pos) if top_pos else "None obvious"),
                        "Top features decreasing default risk: " + (", ".join(top_neg) if top_neg else "None obvious"),
                        "Suggested action: For high-risk cases consider manual review; for low-risk cases consider automated approval with monitoring."
                    ]
                    local_narratives.append("\n".join(narrative_lines))
                except Exception as e:
                    print("Local SHAP explanation failed for index", idx, e)

        except Exception as e:
            print("SHAP analysis error (continuing):", e)

    # -------- Write textual report (metrics + SHAP) --------
    report_lines = []
    report_lines.append("## Credit Risk Model Report\n")
    report_lines.append("### Metrics\n")
    for k, v in metrics.items():
        report_lines.append(f"- {k}: {v}")

    report_lines.append("\n### Best hyperparameters (RandomizedSearchCV)\n")
    report_lines.append(json.dumps(rs.best_params_, indent=2))

    report_lines.append("\n### SHAP Analysis\n")
    if shap_ok and shap_text_lines:
        report_lines.append("\n".join(shap_text_lines))
    else:
        report_lines.append("- SHAP analysis not generated (shap not installed or failed).")

    report_lines.append("\n### Local SHAP narratives (two examples)\n")
    if shap_ok and local_narratives:
        for ln in local_narratives:
            report_lines.append(ln)
            report_lines.append("")  # blank line between narratives
    else:
        report_lines.append("- Local SHAP narratives not generated.")

    with open(OUT_DIR / "report.md", "w") as f:
        f.write("\n".join(report_lines))
    print("Report saved.")

    # -------- Package required files into zip --------
    files_to_include = []
    for p in ["metrics.json", "predictions.csv", "model_pipeline.joblib", "best_params.json", "report.md"]:
        fp = OUT_DIR / p
        if fp.exists():
            files_to_include.append(fp)

    for png in sorted(PLOTS_DIR.glob("*.png")):
        files_to_include.append(png)

    temp_pkg_dir = OUT_DIR / "submission_pkg"
    if temp_pkg_dir.exists():
        shutil.rmtree(temp_pkg_dir)
    temp_pkg_dir.mkdir()

    for fpath in files_to_include:
        shutil.copy(fpath, temp_pkg_dir / fpath.name)

    if PACKAGE_ZIP.exists():
        PACKAGE_ZIP.unlink()
    shutil.make_archive(str(PACKAGE_ZIP.with_suffix('')), 'zip', root_dir=temp_pkg_dir)
    print("Created submission package:", PACKAGE_ZIP)

    total_time = time.time() - t0_total
    print(f"Done in {total_time:.1f}s. Outputs: {OUT_DIR}; Submission ZIP: {PACKAGE_ZIP}")


if __name__ == "__main__":
    main()


Loaded: (32581, 12)
Using target: Default
Train/test: (26064, 13) (6517, 13)
Numeric cols: 10  Categorical cols: 3
XGBoost available.
Running RandomizedSearchCV ...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Search done. Best CV roc_auc: 1.0
Metrics saved.
SHAP available. Running SHAP analysis (may be slow).
Report saved.
Created submission package: /mnt/data/submission_package.zip
Done in 36.2s. Outputs: /mnt/data/outputs; Submission ZIP: /mnt/data/submission_package.zip


<Figure size 600x400 with 0 Axes>

<Figure size 800x950 with 0 Axes>

<Figure size 750x500 with 0 Axes>

<Figure size 750x500 with 0 Axes>

<Figure size 750x500 with 0 Axes>

<Figure size 750x500 with 0 Axes>

<Figure size 800x650 with 0 Axes>